# VisionGuard AI — Phase 3.4: YOLOv8 Nano Training & Evaluation

**Project:** VisionGuard AI — Infrastructure Road Damage Inspection Platform  
**Phase:** Phase 3.4 — Model Training, Validation, and Held-Out Test Evaluation  
**Target Model:** YOLOv8 Nano (`yolov8n.pt`)  
**Dataset:** RDD2022 India Subset (7,706 images: 5,356 train, 1,150 val, 1,200 test)  
**Target Classes:** `0: D00_longitudinal_crack`, `1: D10_transverse_crack`, `2: D20_alligator_crack`, `3: D40_pothole`  
**Execution Platform:** Google Colab (Cloud GPU Runtime)  

---

### Instructions for Running in Google Colab
1. **Select GPU Runtime:** Go to **Runtime** > **Change runtime type**, select **T4 GPU** under Hardware accelerator, and click **Save**.
2. **Verify Dataset:** Step 2.5 and Step 3 will diagnose the backup archive and safely restore the prepared dataset into `/content/visionguard_dataset`.
3. **Execute Sequentially:** Run cells from top to bottom. Do not skip validation cells.
4. **Quarantine Test Split:** The held-out test split is strictly evaluated in Step 9 **only after** training and validation checkpoint selection are finalized.

## Step 1: Runtime & Hardware Verification

Verify GPU model, available VRAM, CUDA capability, and ephemeral disk space in the current Google Colab environment.

In [ ]:
# 1. Check GPU hardware and VRAM allocation
import os
import shutil
import subprocess
import torch

print("=== Google Colab GPU & System Diagnostics ===")
!nvidia-smi

cuda_available = torch.cuda.is_available()
print(f"\nPyTorch CUDA Available: {cuda_available}")

if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    total_vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    print(f"Active GPU Device:      {gpu_name}")
    print(f"Total Dedicated VRAM:   {total_vram_gb} GB")
    if total_vram_gb < 10.0:
        print("WARNING: VRAM is below 10 GB. If training runs Out-Of-Memory, reduce batch size to 4 or 8.")
else:
    print("\n[CRITICAL WARNING] No GPU detected! Please configure Colab:")
    print("  -> Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")

total_b, used_b, free_b = shutil.disk_usage('/content')
print(f"Available Disk Space:   {free_b / (1024**3):.2f} GB free in /content")

## Step 2: Dependency Verification & Ultralytics Installation

Install and verify the official `ultralytics` package and essential computer vision libraries.

In [ ]:
# 2. Install and verify Ultralytics and dependencies
%pip install -q --upgrade ultralytics opencv-python pillow matplotlib pyyaml pandas

import sys
import yaml
import cv2
import PIL
import matplotlib
import ultralytics
from ultralytics import YOLO

print("=== Environment & Library Baseline ===")
print(f"Python Version:      {sys.version.split()[0]}")
print(f"PyTorch Version:     {torch.__version__}")
print(f"Ultralytics Version: {ultralytics.__version__}")
print(f"OpenCV Version:      {cv2.__version__}")
print(f"Pillow Version:      {PIL.__version__}")
print(f"Matplotlib Version:  {matplotlib.__version__}")
print("[PASSED] Dependencies verified.")

## Step 2.5: Non-Destructive Backup Archive Diagnostics

Inspect the Google Drive backup ZIP file **purely in-memory** without extracting everything. This checks the top-level entries, identifies where `data.yaml` is located inside the archive, and detects any nested folder prefix (e.g. `content/visionguard_dataset/`).

In [ ]:
# 2.5 Non-destructive diagnostic check of Google Drive backup archive
import os
import zipfile
from google.colab import drive

print("=== Google Drive Backup Diagnostic ===")
drive.mount('/content/drive')

# Search candidate backup paths
candidate_backups = [
    '/content/drive/MyDrive/VisionGuard_Artifacts/visionguard_rdd2022_india_yolo.zip',
    '/content/drive/MyDrive/VisionGuard_Artifacts/visionguard_rdd2022_japan_yolo.zip',
    '/content/visionguard_rdd2022_india_yolo.zip',
    '/content/visionguard_rdd2022_japan_yolo.zip'
]

backup_zip_path = None
for p in candidate_backups:
    if os.path.exists(p):
        backup_zip_path = p
        break

if backup_zip_path is None:
    # Broad search in Drive folder
    artifacts_dir = '/content/drive/MyDrive/VisionGuard_Artifacts'
    if os.path.exists(artifacts_dir):
        for f in os.listdir(artifacts_dir):
            if f.endswith('.zip') and ('india' in f.lower() or 'rdd' in f.lower() or 'yolo' in f.lower()):
                backup_zip_path = os.path.join(artifacts_dir, f)
                break

assert backup_zip_path is not None, "No backup ZIP found in Google Drive! Please verify your Google Drive folder."

print(f"Found backup archive: {backup_zip_path} ({round(os.path.getsize(backup_zip_path) / 1e6, 2)} MB)")

with zipfile.ZipFile(backup_zip_path, 'r') as zf:
    all_names = zf.namelist()
    print(f"Total entries in backup ZIP: {len(all_names):,}")
    
    # Get top-level directories
    top_levels = sorted(list(set(n.split('/')[0] for n in all_names if '/' in n and not n.startswith('__MACOSX'))))
    print(f"Top-level directory entries:  {top_levels}")
    
    # Search for data.yaml
    yaml_entries = [n for n in all_names if n.endswith('data.yaml') or n.endswith('.yaml')]
    print(f"Discovered YAML files in ZIP: {yaml_entries}")
    
    # Sample member paths
    print("\nFirst 5 path entries in ZIP:")
    for n in all_names[:5]:
        print(f"  - {n}")
        
    if yaml_entries:
        target_yaml_in_zip = yaml_entries[0]
        prefix = os.path.dirname(target_yaml_in_zip)
        print(f"\nDetected dataset path prefix inside ZIP: '{prefix}'")
        if prefix.startswith('content/'):
            print("  -> Notice: Archive contains 'content/' prefix (created via 'zip -r ... /content/visionguard_dataset').")
            print("  -> Step 3 will safely relocate the extracted dataset to '/content/visionguard_dataset'.")

## Step 3: Resilient Dataset Restoration, Normalization & Split Verification

Safely restore the dataset from the Google Drive backup, handle any nested `/content/content/` folder prefixes, consolidate the canonical dataset at `/content/visionguard_dataset`, ensure `data.yaml` has the correct path, and verify that all train/val/test images and labels are intact.

In [ ]:
# 3. Resilient restoration, path normalization, and verification
import os
import shutil
import zipfile
import yaml

CANONICAL_DIR = '/content/visionguard_dataset'
YAML_PATH = os.path.join(CANONICAL_DIR, 'data.yaml')

# Function to search for data.yaml anywhere under /content (excluding /content/drive)
def find_existing_yaml():
    for root, dirs, files in os.walk('/content'):
        if root.startswith('/content/drive'): continue
        if 'data.yaml' in files:
            return os.path.join(root, 'data.yaml')
    return None

existing_yaml = find_existing_yaml()

if existing_yaml is None:
    print(f"Extracting dataset from backup: {backup_zip_path} ...")
    with zipfile.ZipFile(backup_zip_path, 'r') as zf:
        # Check if archive entries start with 'content/'
        has_content_prefix = any(n.startswith('content/') for n in zf.namelist()[:20])
        if has_content_prefix:
            print("Archive entries start with 'content/'. Extracting to root '/' ...")
            zf.extractall('/')
        else:
            print("Extracting directly into /content ...")
            zf.extractall('/content')
    print("Extraction completed.")
    existing_yaml = find_existing_yaml()

assert existing_yaml is not None, "Failed to locate data.yaml after extraction!"
extracted_root = os.path.dirname(existing_yaml)
print(f"Discovered dataset root at: {extracted_root}")

# If dataset was extracted into a nested directory (e.g. /content/content/visionguard_dataset),
# consolidate it cleanly into CANONICAL_DIR (/content/visionguard_dataset)
if os.path.abspath(extracted_root) != os.path.abspath(CANONICAL_DIR):
    print(f"Relocating dataset from {extracted_root} to {CANONICAL_DIR} ...")
    if os.path.exists(CANONICAL_DIR):
        shutil.rmtree(CANONICAL_DIR)
    shutil.move(extracted_root, CANONICAL_DIR)
    
    # Clean up empty parent directories (e.g. /content/content) if left behind
    parent_dir = os.path.dirname(extracted_root)
    if os.path.basename(parent_dir) == 'content' and parent_dir != '/content':
        if len(os.listdir(parent_dir)) == 0:
            os.rmdir(parent_dir)

# Verify canonical YAML file exists
assert os.path.exists(YAML_PATH), f"Canonical YAML file not found at {YAML_PATH}!"

# Load and update path in data.yaml to guarantee exact resolution
with open(YAML_PATH, 'r') as f:
    data_config = yaml.safe_load(f)

data_config['path'] = CANONICAL_DIR
with open(YAML_PATH, 'w') as f:
    yaml.safe_dump(data_config, f, default_flow_style=False)

print(f"\n[PASSED] Dataset established at canonical path: {CANONICAL_DIR}")
print(f"Configuration file verified: {YAML_PATH}")
print("\n=== Parsed data.yaml Content ===")
print(yaml.dump(data_config, default_flow_style=False))

# Verify classes
EXPECTED_CLASSES = {
    0: 'D00_longitudinal_crack',
    1: 'D10_transverse_crack',
    2: 'D20_alligator_crack',
    3: 'D40_pothole'
}
assert data_config['names'] == EXPECTED_CLASSES, f"Class mismatch! Expected {EXPECTED_CLASSES}, got {data_config['names']}"
print("[PASSED] All 4 target defect classes verified.")

# Verify image and label directories and counts
split_images = {}
split_labels = {}

for split in ['train', 'val', 'test']:
    img_dir = os.path.join(CANONICAL_DIR, 'images', split)
    lbl_dir = os.path.join(CANONICAL_DIR, 'labels', split)
    assert os.path.isdir(img_dir), f"Missing image directory: {img_dir}"
    assert os.path.isdir(lbl_dir), f"Missing label directory: {lbl_dir}"
    
    imgs = set(os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
    lbls = set(os.path.splitext(f)[0] for f in os.listdir(lbl_dir) if f.lower().endswith('.txt'))
    
    split_images[split] = imgs
    split_labels[split] = lbls
    print(f"  Split '{split:<5}': {len(imgs):>5,} images | {len(lbls):>5,} labels")

# Verify exact India subset counts
assert len(split_images['train']) == 5356, f"Expected 5,356 train images, found {len(split_images['train'])}"
assert len(split_images['val']) == 1150, f"Expected 1,150 val images, found {len(split_images['val'])}"
assert len(split_images['test']) == 1200, f"Expected 1,200 test images, found {len(split_images['test'])}"

# Assert disjoint splits (zero data leakage)
assert len(split_images['train'] & split_images['val']) == 0, "Leakage: Train and Val overlap!"
assert len(split_images['train'] & split_images['test']) == 0, "Leakage: Train and Test overlap!"
assert len(split_images['val'] & split_images['test']) == 0, "Leakage: Val and Test overlap!"

print("\n[PASSED] All 7,706 images intact, paired with labels, and confirmed strictly disjoint.")
print("[SECURITY] Held-out test set (1,200 images) confirmed isolated.")

## Step 4: Class Imbalance Audit (Pre-Training Distribution)

Audit instance tallies across the training split to establish baseline expectations for each damage category, with special attention to `D10` (transverse cracks) scarcity.

In [ ]:
# 4. Detailed class distribution audit across train split
from collections import defaultdict

train_lbl_dir = os.path.join(CANONICAL_DIR, 'labels', 'train')
class_counts = defaultdict(int)
image_counts = defaultdict(int)
background_images = 0

for lbl_file in os.listdir(train_lbl_dir):
    if not lbl_file.endswith('.txt'): continue
    path = os.path.join(train_lbl_dir, lbl_file)
    with open(path, 'r') as f:
        lines = f.readlines()
    
    if len(lines) == 0:
        background_images += 1
    else:
        seen_in_img = set()
        for line in lines:
            parts = line.strip().split()
            if len(parts) == 5:
                cid = int(parts[0])
                class_counts[cid] += 1
                seen_in_img.add(cid)
        for cid in seen_in_img:
            image_counts[cid] += 1

class_names = data_config['names']
total_objects = sum(class_counts.values())

print("=== Training Split Distress Distribution Audit ===")
print(f"Total Training Images:     {len(os.listdir(train_lbl_dir)):,}")
print(f"Images with Damage:       {len(os.listdir(train_lbl_dir)) - background_images:,}")
print(f"Sound / Background Images: {background_images:,}")
print(f"Total Bounding Boxes:     {total_objects:,}\n")

print(f"{'Class ID':<10} | {'Class Name':<25} | {'Instances':<12} | {'% Share':<10} | {'Images':<10}")
print("-" * 75)
for cid in sorted(class_names.keys()):
    cname = class_names[cid]
    count = class_counts[cid]
    pct = (count / total_objects * 100) if total_objects > 0 else 0
    n_imgs = image_counts[cid]
    print(f"{cid:<10} | {cname:<25} | {count:<12,} | {pct:>6.2f}%    | {n_imgs:<10,}")
print("-" * 75)

# Explicit notice on class imbalance
print("\n[CRITICAL NOTE ON CLASS IMBALANCE]:")
print(" - Potholes (D40) and Alligator cracks (D20) comprise ~76% of all training instances.")
print(" - Transverse cracks (D10) comprise <1.5% of instances. Its recall will be naturally constrained.")
print(" - We will evaluate each class independently to avoid masking D10 performance behind overall mAP.")

## Step 5: Model Initialization & Configuration

Initialize YOLOv8 Nano (`yolov8n.pt`) with pretrained weights for transfer learning, configure hyperparameter settings (`imgsz=640`, `batch=8`, `epochs=30`, `seed=42`, `patience=10`), and prepare experiment logging directories.

In [ ]:
# 5. Initialize model and set hyperparameters
from ultralytics import YOLO

EXPERIMENT_PROJECT = '/content/visionguard_runs'
EXPERIMENT_NAME = 'yolov8n_rdd_india_exp1'

os.makedirs(EXPERIMENT_PROJECT, exist_ok=True)

print("Initializing YOLOv8 Nano baseline...")
model = YOLO('yolov8n.pt')

# Documented Training Configuration
TRAINING_CONFIG = {
    'data': YAML_PATH,
    'epochs': 30,             # Training budget
    'imgsz': 640,             # Standard resolution
    'batch': 8,               # Memory-safe batch size for Colab T4
    'patience': 10,           # Early stopping patience
    'seed': 42,               # Exact reproducibility
    'device': 0 if torch.cuda.is_available() else 'cpu',
    'workers': 2,
    'project': EXPERIMENT_PROJECT,
    'name': EXPERIMENT_NAME,
    'exist_ok': True,
    'verbose': True,
    'plots': True,
    'save': True,
    'val': True               # Validate every epoch for checkpoint selection
}

print("\n=== Planned Training Hyperparameters ===")
for k, v in TRAINING_CONFIG.items():
    print(f"  {k:<12}: {v}")

## Step 6: Training Execution

Execute training on the training split while monitoring loss convergence and validation metrics. The validation set is used for periodic evaluation and best-checkpoint selection (`best.pt`).

In [ ]:
# 6. Execute YOLOv8 Nano training
import time

start_time = time.time()
print(f"Starting YOLOv8 Nano training at {time.strftime('%Y-%m-%d %H:%M:%S')}...")
print("This process typically takes 30–50 minutes on a Tesla T4 GPU.\n")

train_results = model.train(**TRAINING_CONFIG)

elapsed_min = (time.time() - start_time) / 60.0
print(f"\n[COMPLETED] Training finished in {elapsed_min:.2f} minutes.")

RUN_DIR = os.path.join(EXPERIMENT_PROJECT, EXPERIMENT_NAME)
BEST_WEIGHTS_PATH = os.path.join(RUN_DIR, 'weights', 'best.pt')
LAST_WEIGHTS_PATH = os.path.join(RUN_DIR, 'weights', 'last.pt')

assert os.path.exists(BEST_WEIGHTS_PATH), f"Checkpoint not found at {BEST_WEIGHTS_PATH}!"
print(f"Best model weights saved at: {BEST_WEIGHTS_PATH} ({round(os.path.getsize(BEST_WEIGHTS_PATH) / 1e6, 2)} MB)")

## Step 7: Validation Performance & Loss Curve Diagnostics

Inspect the generated training and validation loss curves, mAP progression, and confusion matrix from the completed training run.

In [ ]:
# 7. Render training curves and validation diagnostics
import matplotlib.pyplot as plt
from PIL import Image

results_png = os.path.join(RUN_DIR, 'results.png')
confusion_matrix_png = os.path.join(RUN_DIR, 'confusion_matrix.png')

if os.path.exists(results_png):
    plt.figure(figsize=(16, 10))
    plt.imshow(Image.open(results_png))
    plt.title("YOLOv8 Training and Validation Loss Curves", fontsize=14)
    plt.axis('off')
    plt.show()
else:
    print("results.png not found.")

if os.path.exists(confusion_matrix_png):
    plt.figure(figsize=(10, 10))
    plt.imshow(Image.open(confusion_matrix_png))
    plt.title("Validation Confusion Matrix", fontsize=14)
    plt.axis('off')
    plt.show()
else:
    print("confusion_matrix.png not found.")

## Step 8: Validation Metrics & Class-Wise Performance Analysis

Run formal validation of `best.pt` on the validation set, extracting precision, recall, mAP@0.5, and mAP@0.5:0.95 across each individual damage class.

In [ ]:
# 8. Formal validation evaluation of best checkpoint
best_model = YOLO(BEST_WEIGHTS_PATH)

print("Running validation evaluation on validation split...")
val_metrics = best_model.val(
    data=YAML_PATH,
    split='val',
    imgsz=640,
    batch=8,
    conf=0.25,
    iou=0.6,
    project=EXPERIMENT_PROJECT,
    name='validation_eval',
    exist_ok=True
)

print("\n" + "="*75)
print("           VALIDATION SPLIT EVALUATION REPORT (best.pt)")
print("="*75)
print(f"Mean Precision:     {val_metrics.box.mp:.4f}")
print(f"Mean Recall:        {val_metrics.box.mr:.4f}")
print(f"mAP@0.5:            {val_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95:       {val_metrics.box.map:.4f}")
print("-"*75)
print(f"{'Class ID':<10} | {'Class Name':<25} | {'Precision':<12} | {'Recall':<10} | {'mAP@0.5':<10}")
print("-"*75)

for i, cid in enumerate(sorted(class_names.keys())):
    cname = class_names[cid]
    p = val_metrics.box.p[i] if i < len(val_metrics.box.p) else 0.0
    r = val_metrics.box.r[i] if i < len(val_metrics.box.r) else 0.0
    ap50 = val_metrics.box.ap50[i] if i < len(val_metrics.box.ap50) else 0.0
    print(f"{cid:<10} | {cname:<25} | {p:<12.4f} | {r:<10.4f} | {ap50:<10.4f}")
print("="*75)

## Step 9: Final Evaluation on the Held-Out Test Set

> [!IMPORTANT]
> **Strict Protocol:** The held-out test set (`1,200 images`) is evaluated **only now**, after model weights and hyperparameters are frozen. It was never seen during training or checkpoint selection.

In [ ]:
# 9. Formal evaluation on held-out test split
print("Running final evaluation on quarantined held-out TEST split...")
test_metrics = best_model.val(
    data=YAML_PATH,
    split='test',
    imgsz=640,
    batch=8,
    conf=0.25,
    iou=0.6,
    project=EXPERIMENT_PROJECT,
    name='test_eval',
    exist_ok=True
)

print("\n" + "="*75)
print("           HELD-OUT TEST SET EVALUATION REPORT (best.pt)")
print("="*75)
print(f"Mean Precision:     {test_metrics.box.mp:.4f}")
print(f"Mean Recall:        {test_metrics.box.mr:.4f}")
print(f"mAP@0.5:            {test_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95:       {test_metrics.box.map:.4f}")
print("-"*75)
print(f"{'Class ID':<10} | {'Class Name':<25} | {'Precision':<12} | {'Recall':<10} | {'mAP@0.5':<10}")
print("-"*75)

for i, cid in enumerate(sorted(class_names.keys())):
    cname = class_names[cid]
    p = test_metrics.box.p[i] if i < len(test_metrics.box.p) else 0.0
    r = test_metrics.box.r[i] if i < len(test_metrics.box.r) else 0.0
    ap50 = test_metrics.box.ap50[i] if i < len(test_metrics.box.ap50) else 0.0
    print(f"{cid:<10} | {cname:<25} | {p:<12.4f} | {r:<10.4f} | {ap50:<10.4f}")
print("="*75)

## Step 10: Qualitative Predictions Visualization on Test Images

Run inference on sample test images and render predicted bounding boxes, defect labels, and detection confidence scores.

In [ ]:
# 10. Visualize predictions on sample test images
test_img_dir = os.path.join(CANONICAL_DIR, 'images', 'test')
sample_test_files = [f for f in os.listdir(test_img_dir) if f.lower().endswith(('.jpg', '.png'))][:8]

plt.figure(figsize=(16, 8))
for i, s_file in enumerate(sample_test_files):
    img_path = os.path.join(test_img_dir, s_file)
    preds = best_model.predict(source=img_path, conf=0.25, verbose=False)[0]
    annotated_img = preds.plot()
    annotated_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
    
    plt.subplot(2, 4, i + 1)
    plt.imshow(annotated_rgb)
    plt.title(s_file, fontsize=9)
    plt.axis('off')

plt.tight_layout()
plt.show()

## Step 11: Model Checkpoint Export & Google Drive Backup

Save the best checkpoint (`best.pt`), export evaluation summaries to JSON, and back up the weights to Google Drive.

In [ ]:
# 11. Save model weights and summary to Google Drive
import json

DRIVE_ARTIFACTS_DIR = '/content/drive/MyDrive/VisionGuard_Artifacts'
os.makedirs(DRIVE_ARTIFACTS_DIR, exist_ok=True)

# Export summary JSON
summary = {
    'model': 'YOLOv8 Nano (yolov8n.pt)',
    'dataset': 'RDD2022 India Subset',
    'epochs_trained': 30,
    'image_size': 640,
    'batch_size': 8,
    'validation_metrics': {
        'mean_precision': float(val_metrics.box.mp),
        'mean_recall': float(val_metrics.box.mr),
        'mAP_50': float(val_metrics.box.map50),
        'mAP_50_95': float(val_metrics.box.map)
    },
    'test_metrics': {
        'mean_precision': float(test_metrics.box.mp),
        'mean_recall': float(test_metrics.box.mr),
        'mAP_50': float(test_metrics.box.map50),
        'mAP_50_95': float(test_metrics.box.map)
    },
    'classes': data_config['names']
}

summary_json_path = os.path.join(RUN_DIR, 'experiment_summary.json')
with open(summary_json_path, 'w') as f:
    json.dump(summary, f, indent=2)

# Copy best.pt to Drive
drive_best_pt = os.path.join(DRIVE_ARTIFACTS_DIR, 'best.pt')
drive_summary = os.path.join(DRIVE_ARTIFACTS_DIR, 'experiment_summary.json')

shutil.copyfile(BEST_WEIGHTS_PATH, drive_best_pt)
shutil.copyfile(summary_json_path, drive_summary)

print(f"[SUCCESS] Best weights backed up to Drive: {drive_best_pt} ({round(os.path.getsize(drive_best_pt) / 1e6, 2)} MB)")
print(f"[SUCCESS] Summary backed up to Drive:      {drive_summary}")

print("\n--- HOW TO DOWNLOAD TO LOCAL LAPTOP FOR PHASE 4 FLASK INTEGRATION ---")
print("Option A: Open Google Drive in your browser, navigate to 'VisionGuard_Artifacts', and download 'best.pt'.")
print("Option B: Run in Colab: from google.colab import files; files.download(BEST_WEIGHTS_PATH)")
print("Place the downloaded 'best.pt' file into your local project at: visionGuard/ai/models/best.pt")

## Step 12: Phase 3.4 Completion & Engineering Guardrails Notice

> [!CAUTION]
> **Engineering Guardrails:**  
> 1. A 2D bounding box localizes distress instances and calculates pixel dimensions, but **does not measure true physical crack width** without camera calibration and ground-plane homography.  
> 2. Statistical model confidence measures detection certainty, **not structural safety or severity**.  
> 3. Transverse crack (`D10`) metrics reflect real-world data scarcity in the Indian road network.